In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler, TensorDataset
from torch.nn.utils import clip_grad_norm_
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import RobertaTokenizer, RobertaForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm
import numpy as np
import random
import os




In [2]:
# Ensure deterministic behavior
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


In [3]:
# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [4]:
# Load 20 Newsgroups dataset
newsgroups_data = fetch_20newsgroups(subset='train')


In [5]:
# Assign directly to train_texts and train_labels
train_texts = [text for text in newsgroups_data.data if text.strip() != ""]
train_labels = [label for text, label in zip(newsgroups_data.data, newsgroups_data.target) if text.strip() != ""]


In [6]:
# Load RoBERTa tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

In [7]:
# Tokenize in chunks to reduce memory usage
max_length = 512
batch_size = 500
input_ids, attention_masks = [], []
for i in range(0, len(train_texts), batch_size):
    batch = tokenizer(train_texts[i:i+batch_size], padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    input_ids.append(batch['input_ids'])
    attention_masks.append(batch['attention_mask'])

train_input_ids = torch.cat(input_ids, dim=0)
train_attention_mask = torch.cat(attention_masks, dim=0)
train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)


In [8]:
# Dataset and Dataloader
train_dataset = TensorDataset(train_input_ids, train_attention_mask, train_labels_tensor)
train_dataloader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=8, num_workers=0)


In [9]:
# Load RoBERTa model
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=20)
model.to(device)


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [10]:
# Optimizer and Scheduler
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
epochs = 3
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)


In [11]:
# Training loop
def train():
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

        for batch in progress_bar:
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            model.zero_grad()
            outputs = model(input_ids=b_input_ids, attention_mask=b_input_mask, labels=b_labels)
            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()
            clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            progress_bar.set_postfix({"loss": loss.item()})

        avg_loss = total_loss / len(train_dataloader)
        print(f"Epoch {epoch+1} finished. Average Loss: {avg_loss:.4f}")

print("RoBERTa setup complete. Run train() to begin training.")


RoBERTa setup complete. Run train() to begin training.


In [12]:
train()

Epoch 1 finished. Average Loss: 0.8856


Epoch 2 finished. Average Loss: 0.3261


Epoch 3 finished. Average Loss: 0.1776


In [13]:
# Save model and tokenizer
model_path = "./roberta_model"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy to Google Drive
!mkdir -p /content/drive/MyDrive/roberta_models
!cp -r ./roberta_model /content/drive/MyDrive/roberta_models/

print(" Model and tokenizer saved to: /content/drive/MyDrive/roberta_models/roberta_model")


Mounted at /content/drive
 Model and tokenizer saved to: /content/drive/MyDrive/roberta_models/roberta_model


In [15]:
from google.colab import drive
drive.mount('/content/drive')


from transformers import RobertaTokenizer, RobertaForSequenceClassification
import os

model_path = "/content/drive/MyDrive/roberta_models/roberta_model"

model = RobertaForSequenceClassification.from_pretrained(model_path, local_files_only=True)
roberta_tokenizer = RobertaTokenizer.from_pretrained(model_path, local_files_only=True)
model.to(device) # Add this line to move the model to the correct device

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [17]:
from sklearn.datasets import fetch_20newsgroups

test_data = fetch_20newsgroups(subset='test')
test_texts = test_data.data
test_labels = test_data.target


In [18]:
# Tokenize test data
test_encodings = roberta_tokenizer(test_texts, padding=True, truncation=True, max_length=512)
test_input_ids = torch.tensor(test_encodings['input_ids'])
test_attention_mask = torch.tensor(test_encodings['attention_mask'])
test_labels_tensor = torch.tensor(test_labels)


In [19]:
# Dataset and Dataloader
test_dataset = TensorDataset(test_input_ids, test_attention_mask, test_labels_tensor)
test_dataloader = DataLoader(test_dataset, batch_size=8, sampler=SequentialSampler(test_dataset))


In [20]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

def evaluate_test_roberta():
    model.eval()
    preds = []
    true_labels = []

    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Evaluating RoBERTa"):
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            outputs = model(b_input_ids, attention_mask=b_input_mask)
            logits = outputs.logits

            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            true_labels.extend(b_labels.cpu().numpy())

    # Calculate metrics
    acc = accuracy_score(true_labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, preds, average='weighted')

    print("\n RoBERTa Evaluation Completed")
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")


In [21]:
evaluate_test_roberta()

Evaluating RoBERTa: 100%|██████████| 942/942 [03:53<00:00,  4.04it/s]


 RoBERTa Evaluation Completed
Test Accuracy: 0.8597
Precision: 0.8626, Recall: 0.8597, F1 Score: 0.8601
